In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data_dir = r"D:\ForestFire\CBH\data"
df_train = pd.read_csv(os.path.join(data_dir, 'NFI6-7_cleaned2.csv'), encoding='cp949')

In [ ]:
df_train.info()

In [ ]:
from matplotlib import rc
from scipy.stats import shapiro, skew, kstest, kurtosis
rc('font', family='NanumGothic') 

In [ ]:
# Data Inspection
# sample size
sample_size =  pd.DataFrame(df_train['SID'].value_counts()).reset_index()
sample_size['var'] = [0.] * len(sample_size)
sample_size['min'] = [0.] * len(sample_size)
sample_size['max'] = [0.] * len(sample_size)
# variance per species
unique_sid = df_train['SID'].unique()
n_row = 8
n_col = 5
fig, axes = plt.subplots(n_row, n_col, figsize=(15, 12))
axes = axes.flatten()
for i, sid in enumerate(unique_sid):
    cr_data = df_train.loc[df_train['SID'] == sid, 'CR']
    s_name = df_train.loc[df_train['SID'] == sid, 'I_Species'].unique()[0]
    s_size = sample_size.loc[sample_size['SID'] == sid, 'count'].values[0] # or .numpy()[0]
    sample_size.loc[sample_size['SID'] == sid, 'var'] = np.var(cr_data)
    sample_size.loc[sample_size['SID'] == sid, 'min'] = np.min(cr_data)
    sample_size.loc[sample_size['SID'] == sid, 'max'] = np.max(cr_data)
    std_val = np.std(cr_data)
    skew_val = skew(cr_data)
    kurt_val = kurtosis(cr_data)
    q1 = np.quantile(cr_data, 0.25)
    q2 = np.quantile(cr_data, 0.5)
    q3 = np.quantile(cr_data, 0.75)
    # 정규성 검정
    if len(cr_data) >= 30:
        stat, p_val = kstest(cr_data, 'norm')
        p_val_str = f"p_val: {p_val:.3f}"
    else:
        p_val_str = "N/A"
    ax = axes[i]
    ax.hist(cr_data, bins=10)
    ax.axvline(x=q1, color='r', linestyle='--')
    ax.axvline(x=q2, color='r', linestyle='--')
    ax.axvline(x=q3, color='r', linestyle='--')
    ax.set_title(f"Histogram of {s_name}({s_size})", fontsize=8)
    # legend 추가: 표준편차, 왜도, 정규성 검정의 p_value
    legend_text = f"n = {s_size}, std = {std_val:.3f}\nskew = {skew_val:.2f}, kurt = {kurt_val:.3f}\nk-s test {p_val_str}\nmin = {np.min(cr_data):.2f}, max = {np.max(cr_data):.2f}"
    ax.legend([legend_text], loc="upper right", fontsize=6, handlelength=0)

# delete empty axes
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

# save figure
fig_dir = r"D:\ForestFire\CBH\fig"
plt.savefig(os.path.join(fig_dir, "histogram-crownratio-species.png"))

# show figure
plt.show()

In [ ]:
sample_size

In [ ]:
df_train.columns

In [ ]:
# variance of the independant variables
sample_size['var_h'] = [0.] * len(sample_size)
sample_size['var_dbh'] = [0.] * len(sample_size)
sample_size['var_cd'] = [0.] * len(sample_size)
sample_size['var_el'] = [0.] * len(sample_size)
sample_size['var_sl'] = [0.] * len(sample_size)
sample_size['var_az'] = [0.] * len(sample_size)
for i, sid in enumerate(unique_sid):
    s_data = df_train.loc[df_train['SID'] == sid, :]
    s_name = df_train.loc[df_train['SID'] == sid, 'I_Species'].unique()[0]
    s_size = sample_size.loc[sample_size['SID'] == sid, 'count'].values[0] # or .numpy()[0]
    sample_size.loc[sample_size['SID'] == sid, 'var_h'] = np.std(s_data['H(ft)'])
    sample_size.loc[sample_size['SID'] == sid, 'var_dbh'] = np.std(s_data['DBH(inch)'])
    sample_size.loc[sample_size['SID'] == sid, 'var_cd'] = np.std(s_data['CD(%)'])
    sample_size.loc[sample_size['SID'] == sid, 'var_el'] = np.std(s_data['Elev(hm)'])
    sample_size.loc[sample_size['SID'] == sid, 'var_sl'] = np.std(s_data['Slope(tan)'])
    sample_size.loc[sample_size['SID'] == sid, 'var_az'] = np.std(s_data['Azimuth(rad)'])
sample_size

In [ ]:
def drawHistogram(df, col_name, n_row=8, n_col=5):
    sample_size =  pd.DataFrame(df['SID'].value_counts()).reset_index()
    sample_size['var'] = [0.] * len(sample_size)
    sample_size['min'] = [0.] * len(sample_size)
    sample_size['max'] = [0.] * len(sample_size)
    # variance per species
    unique_sid = df['SID'].unique()
    fig, axes = plt.subplots(n_row, n_col, figsize=(18, 15))
    axes = axes.flatten()
    for i, sid in enumerate(unique_sid):
        cr_data = df.loc[df['SID'] == sid, col_name]
        s_name = df_train.loc[df_train['SID'] == sid, 'I_Species'].unique()[0]
        s_size = sample_size.loc[sample_size['SID'] == sid, 'count'].values[0] # or .numpy()[0]
        sample_size.loc[sample_size['SID'] == sid, 'var'] = np.var(cr_data)
        sample_size.loc[sample_size['SID'] == sid, 'min'] = np.min(cr_data)
        sample_size.loc[sample_size['SID'] == sid, 'max'] = np.max(cr_data)
        std_val = np.std(cr_data)
        skew_val = skew(cr_data)
        kurt_val = kurtosis(cr_data)
        q1 = np.quantile(cr_data, 0.25)
        q2 = np.quantile(cr_data, 0.5)
        q3 = np.quantile(cr_data, 0.75)
        # 정규성 검정
        if len(cr_data) >= 30:
            stat, p_val = kstest(cr_data, 'norm')
            p_val_str = f"p_val: {p_val:.3f}"
        else:
            p_val_str = "N/A"
        ax = axes[i]
        ax.hist(cr_data, bins=10)
        ax.axvline(x=q1, color='r', linestyle='--')
        ax.axvline(x=q2, color='r', linestyle='--')
        ax.axvline(x=q3, color='r', linestyle='--')
        ax.set_title(f"Histogram of {s_name}({s_size})", fontsize=8)
        # legend 추가: 표준편차, 왜도, 정규성 검정의 p_value
        legend_text = f"n = {s_size}, std = {std_val:.3f}\nskew = {skew_val:.2f}, kurt = {kurt_val:.3f}\nk-s test {p_val_str}\nmin = {np.min(cr_data):.2f}, max = {np.max(cr_data):.2f}"
        ax.legend([legend_text], loc="upper right", fontsize=6, handlelength=0)

    # delete empty axes
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    plt.suptitle(f"Histogram of {col_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    # save figure
    fig_dir = r"D:\ForestFire\CBH\fig"
    plt.savefig(os.path.join(fig_dir, f"histogram-{col_name}-species.png"))
    
    # show figure
    plt.show()

    return sample_size

In [ ]:
# H(ft), DBH(inch), CD(%)
summary_height = drawHistogram(df_train, "H(ft)")

In [ ]:
# 'Elev(hm)', 'Slope(tan)','Azimuth(rad)'
summary_sl = drawHistogram(df_train, "Azimuth(rad)")

In [ ]:
def drawPairPlot(df, s_name, opt_save, col_list=['DBH(inch)', 'H(ft)', 'CBH(ft)', 'CR', 'CH(ft)', 'Elev(hm)', 'Azimuth(rad)', 'CD(%)'], f_name=None):
    condition = (df['Species'] == s_name)
    df_species = df.loc[condition, col_list]
    sns.pairplot(df_species, kind='reg', plot_kws={'line_kws' : {'color' : 'orange'}})
    if opt_save: 
        save_dir = r'D:/ForestFire/CBH/fig'
        plt.savefig(os.path.join(save_dir, f_name))

In [ ]:
drawPairPlot(df_train, "소나무", opt_save=False)